# 40-Multivariate-Iterative Imputer

src: https://www.youtube.com/watch?v=a38ehxv3kyk

In [24]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [25]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np

# Step 2: Load the dataset and round values for easier readability
# We're using only the relevant 4 columns: R&D Spend, Administration, Marketing Spend, and Profit
# Values are divided by 10,000 to bring them into a manageable range (e.g. 154000 becomes 15.4)
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']] / 10000)

# Step 3: Set a seed so the random sampling is reproducible (you always get the same rows)
np.random.seed(9)

# Step 4: Randomly select 5 rows from the dataset
# This simulates a small sample dataset for demonstration/testing
df = df.sample(5)

# Step 5: Show the selected 5 rows
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [26]:
# Drop the last column (i.e., the 'Profit' column) from the dataset
# We're keeping only the features — R&D Spend, Administration, Marketing Spend
df = df.iloc[:, 0:-1]

# Display the resulting DataFrame
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [27]:
# Use .at[] or .iloc[] carefully to assign NaN values — avoid chained assignment warnings

import numpy as np

# Set R&D Spend at row index 1 (i.e., ID 37) to NaN
df.at[37, 'R&D Spend'] = np.nan

# Set Administration at row index 3 (i.e., ID 14) to NaN
df.at[14, 'Administration'] = np.nan

# Set Marketing Spend at the last row (i.e., ID 44) to NaN
df.at[44, 'Marketing Spend'] = np.nan

# Display the modified DataFrame with NaN values
df


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [28]:
# Step 1: Impute all missing (NaN) values with the mean of their respective column

# Create a new DataFrame to store the imputed result so the original `df` remains unchanged
df0 = pd.DataFrame()

# For 'R&D Spend', fill NaN with the column mean
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())

# For 'Administration', fill NaN with the column mean
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())

# For 'Marketing Spend', fill NaN with the column mean
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

---

### 🧪 Let's Verify the Imputed Values

From the earlier state of `df` with NaNs:

| ID | R\&D Spend | Administration | Marketing Spend |
| -- | ---------- | -------------- | --------------- |
| 21 | 8.0        | 15.0           | 30.0            |
| 37 | NaN        | 5.0            | 20.0            |
| 2  | 15.0       | 10.0           | 41.0            |
| 14 | 12.0       | NaN            | 26.0            |
| 44 | 2.0        | 15.0           | NaN             |

We can calculate the **mean** for each column (ignoring NaNs):

```python
df.mean()
```

Output:

```
R&D Spend         (8 + 15 + 12 + 2) / 4  = 9.25
Administration    (15 + 5 + 10 + 15) / 4 = 11.25
Marketing Spend   (30 + 20 + 41 + 26) / 4 = 29.25
```

So your `df0` should now look like:

| ID | R\&D Spend | Administration | Marketing Spend |
| -- | ---------- | -------------- | --------------- |
| 21 | 8.0        | 15.0           | 30.0            |
| 37 | **9.25**   | 5.0            | 20.0            |
| 2  | 15.0       | 10.0           | 41.0            |
| 14 | 12.0       | **11.25**      | 26.0            |
| 44 | 2.0        | 15.0           | **29.25**       |

✅ This matches the exact `Iteration 0` snapshot from your earlier images.

---


In [29]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


## Step 2 of the MICE process

In **Step 2 of the MICE process**, we:

* **Remove the previously imputed value** (in this case, R\&D Spend for ID=37),
* Then you'll **predict it using a machine learning model**.

---

### 🧪 Output of `df1`

| ID | R\&D Spend | Administration | Marketing Spend |
| -- | ---------- | -------------- | --------------- |
| 21 | 8.00       | 15.0           | 30.0            |
| 37 | **NaN**    | 5.0            | 20.0            |
| 2  | 15.00      | 10.0           | 41.0            |
| 14 | 12.00      | 11.25          | 26.0            |
| 44 | 2.00       | 15.0           | 29.25           |

✅ This matches what we expect: now `R&D Spend` at index `1` (ID=37) is back to missing — ready to be predicted using a regression model on the remaining data.

---

In [30]:
# Step 2: Begin iterative imputation (left-to-right)

# Make a copy of the mean-imputed DataFrame so original (df0) stays unchanged
df1 = df0.copy()

# Set the previously imputed R&D Spend value (row 1) back to NaN
# This simulates "now we try to predict this using other columns"
df1.iloc[1, 0] = np.nan

# Display the DataFrame where first missing value (R&D Spend at row 1) is now to be predicted
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [31]:
# Step 3: Build training input (X) to predict the missing R&D Spend

# From df1, select rows where R&D Spend is not missing — i.e., all except row index 1
# Use columns 1 and 2 → which are 'Administration' and 'Marketing Spend'
# These will be used as independent variables (features)
X = df1.iloc[[0, 2, 3, 4], 1:3]

# Display the training features
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [32]:
# Step 4: Create the target variable `y`
# This is the column we want to predict: R&D Spend
# Select the same 4 rows used in `X` (i.e., rows where R&D is not NaN)
y = df1.iloc[[0, 2, 3, 4], 0]

# Display the target values
y

,R&D Spend
21,8.0
2,15.0
14,12.0
44,2.0


### **Replace a mean-imputed value with a model-based prediction**

In [33]:
from sklearn.linear_model import LinearRegression

# Step 5a: Create a linear regression model instance
lr = LinearRegression()

# Step 5b: Train the model using known X and y values
lr.fit(X, y)

# Step 5c: Predict the missing R&D Spend using row 1's other values
# df1.iloc[1, 1:] → gives the 2 input features: [Administration, Marketing Spend]
# .values.reshape(1, 2) → reshapes it to a 2D array for scikit-learn (required shape: [n_samples, n_features])
lr.predict(df1.iloc[1, 1:].values.reshape(1, 2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [34]:
# Step 6: Insert the predicted value (23.14) into the DataFrame at the missing location
# Row index 1, Column index 0 (i.e., R&D Spend for ID=37)
df1.iloc[1, 0] = 23.14

In [35]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


---

You're now inserting the **predicted value** back into the DataFrame at the position where `R&D Spend` was missing.

---

### 🧪 Resulting DataFrame (`df1`) after First Imputation

| ID | R\&D Spend | Administration | Marketing Spend |
| -- | ---------- | -------------- | --------------- |
| 21 | 8.00       | 15.00          | 30.00           |
| 37 | **23.14**  | 5.00           | 20.00           |
| 2  | 15.00      | 10.00          | 41.00           |
| 14 | 12.00      | **11.25**      | 26.00           |
| 44 | 2.00       | 15.00          | **29.25**       |

🧠 This completes the **first iteration step** of MICE:

* You just replaced a mean-imputed value with a **model-based prediction**.

---

### **continuing with Step 2 for the second column**

This time, you're removing the imputed value in the `Administration` column (row index 3 / ID=14), to **predict it using a regression model** just like you did for `R&D Spend`.

In [36]:
# Step 7: Remove the imputed Administration value at row index 3 (ID=14)
# This will now be predicted using a model trained on the other complete rows
df1.iloc[3, 1] = np.nan

# Show the updated DataFrame with Administration set back to NaN
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


---

🧠 Now you're ready to:

1. Build new `X` (features) using rows where `Administration` is not missing.
2. Create corresponding `y` (target = Administration).
3. Train a regression model.
4. Predict the missing value (row index 3).

---


In [37]:
# Step 8: Create input features `X` to predict the missing Administration value

# Select rows where Administration is NOT missing: rows 0, 1, 2, and 4
# Columns used as predictors: column 0 (R&D Spend) and column 2 (Marketing Spend)
# That is, use (R&D Spend, Marketing Spend) to predict Administration
X = df1.iloc[[0, 1, 2, 4], [0, 2]]

# Display the feature matrix
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


✅ These rows have valid (non-missing) Administration values — suitable for model training.

In [38]:
# Step 9: Create the target variable `y`
# This is the column we want to predict: Administration
# Select the same 4 rows used in X (i.e., where Administration is not missing)
y = df1.iloc[[0, 1, 2, 4], 1]

# Show the target values
y

,Administration
21,15.0
37,5.0
2,10.0
44,15.0


✅ These are the correct corresponding target values for the feature matrix X (R&D Spend and Marketing Spend).

### **Train the Model to predict**

---

Now you're training a model to predict the missing **Administration** value (row index 3) using the columns:

* `R&D Spend` = 12.0
* `Marketing Spend` = 26.0

---

### 🧪 Input Values Being Passed to `predict()`

```python
df1.iloc[3, [0, 2]].values → array([12.0, 26.0])
```

So the model is predicting:

> What would `Administration` be if
> R\&D Spend = 12.0
> Marketing Spend = 26.0?

---

### 🔮 Expected Output

Based on your earlier transcript and image walkthrough, the predicted value should be approximately:

```python
array([11.06])
```

---

In [39]:
from sklearn.linear_model import LinearRegression

# Step 10a: Create a Linear Regression model instance
lr = LinearRegression()

# Step 10b: Train the model on input features X and target y (Administration)
lr.fit(X, y)

# Step 10c: Predict the missing Administration value for row index 3 (ID=14)
# Use R&D Spend and Marketing Spend from row 3 as input
# Reshape is needed to convert it into shape [1, 2] expected by sklearn
lr.predict(df1.iloc[3, [0, 2]].values.reshape(1, 2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.06331285])

In [40]:
df1.iloc[3,1] = 11.06

In [41]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [42]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [43]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [44]:
y = df1.iloc[0:4,-1]
y

,Marketing Spend
21,30.0
37,20.0
2,41.0
14,26.0


In [45]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([31.56351448])

In [46]:
df1.iloc[4,-1] = 31.56

In [47]:
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [48]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [49]:
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [50]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.78627207])

In [51]:
df2.iloc[1,0] = 23.78

In [52]:
df2.iloc[3,1] = np.nan
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.22020174])

In [53]:
df2.iloc[3,1] = 11.22

In [54]:
df2.iloc[4,-1] = np.nan

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([38.87979054])

In [55]:
df2.iloc[4,-1] = 31.56

In [56]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [57]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [58]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [59]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([24.57698058])

In [60]:
df3.iloc[1,0] = 24.57

In [61]:
df3.iloc[3,1] = np.nan
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.37282844])

In [62]:
df3.iloc[3,1] = 11.37

In [63]:
df3.iloc[4,-1] = np.nan

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([45.53976417])

In [64]:
df3.iloc[4,-1] = 45.53

In [65]:
df2.iloc[3,1] = 11.22

In [66]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [67]:
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97
